<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/Ciencia%20De%20Datos%20Con%20Python/CuadernosColab/CienciaDeDatos_Cap%C3%ADtulo015_Eliminar_imputar_o_reconstruir_valores_faltantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 15 — Eliminar, imputar o reconstruir valores faltantes

## Breve repaso

En el trabajo anterior sobre valores faltantes aprendimos a diagnosticar la presencia de datos incompletos dentro de un `DataFrame`.

Vimos que Pandas suele representar los valores faltantes como `NaN`, y usamos herramientas como `isna()`, `sum()` y `mean()` para detectar cuántos faltantes había en cada columna y qué porcentaje representaban sobre el total de filas.

También observamos que no todos los problemas de ausencia aparecen como `NaN`. Algunos datasets usan valores textuales como `"UNKNOWN"` o `"ERROR"` para indicar información desconocida, errores de carga o datos que no pudieron registrarse correctamente. Por eso, además de contar valores faltantes reconocidos por Pandas, también revisamos valores únicos y frecuencias en columnas categóricas.

En este capítulo vamos a avanzar un paso más: vamos a estudiar distintas estrategias para tratar valores faltantes.

La idea no será aplicar una solución automática, sino analizar distintas posibilidades. A veces puede tener sentido eliminar filas incompletas. En otros casos, puede ser mejor conservarlas. Algunas columnas pueden completarse con un valor razonable, mientras que otras pueden reconstruirse usando información disponible en el propio dataset.

Trabajaremos con tres grandes estrategias:

```text
eliminar datos faltantes
imputar valores faltantes
reconstruir valores faltantes a partir de otras columnas
```

Cada estrategia tiene ventajas, riesgos y condiciones de uso. Por eso, el objetivo de este capítulo no es memorizar comandos, sino aprender a decidir con criterio.

Al finalizar este capítulo deberías poder:

* Comprender la diferencia entre eliminar, imputar y reconstruir valores faltantes.
* Usar `dropna()` para eliminar filas o columnas con valores faltantes.
* Usar `fillna()` para completar valores faltantes en algunos casos.
* Evaluar los riesgos de eliminar datos.
* Evaluar los riesgos de imputar datos.
* Reconstruir valores faltantes cuando existe una relación confiable entre columnas.
* Verificar el resultado después de tratar valores faltantes.
* Comprender por qué no hay una única estrategia correcta para todos los datasets.


## Dataset de trabajo

En este capítulo vamos a seguir usando el dataset **Cafe Sales — Dirty Data for Cleaning Training**.

Cada fila representa una transacción de venta en una cafetería. Las columnas incluyen información sobre el producto vendido, la cantidad, el precio unitario, el total gastado, el método de pago, la ubicación y la fecha.

Como vamos a trabajar con valores faltantes, este dataset es especialmente útil porque contiene datos incompletos y valores problemáticos que requieren decisiones de limpieza.

In [ ]:
import kagglehub
import pandas as pd
import os

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

archivos = os.listdir(ruta_dataset)

archivos_csv = [
    archivo for archivo in archivos
    if archivo.endswith(".csv")
]

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df = pd.read_csv(ruta_csv)

df.head()

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Antes de decidir cómo tratar los valores faltantes, conviene recordar el estado general del dataset.

Vamos a revisar nuevamente la cantidad de filas y columnas, los valores faltantes reconocidos por Pandas y algunos valores textuales problemáticos.

In [ ]:
df.shape

(10000, 8)

In [ ]:
df.isna().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


In [ ]:
df[["Item", "Payment Method", "Location"]].apply(
    lambda columna: columna.isin(["UNKNOWN", "ERROR"]).sum()
)

,0
Item,636
Payment Method,599
Location,696


Estas primeras revisiones nos permiten recuperar el diagnóstico inicial.

Tenemos, por un lado, valores faltantes reconocidos por Pandas como `NaN`. Por otro lado, tenemos valores escritos como `"UNKNOWN"` o `"ERROR"` que también pueden representar ausencia de información o errores de carga, aunque Pandas los trate como texto común.

En este capítulo vamos a trabajar principalmente con estrategias de tratamiento. Antes de aplicar cualquier transformación, mantendremos una idea central: cada decisión debe justificarse según el tipo de columna, el problema detectado y el análisis que queremos realizar.

## Tres estrategias posibles

Cuando encontramos valores faltantes, podemos seguir distintas estrategias.

No existe una única solución correcta para todos los casos. La decisión depende del contexto, de la columna afectada, de la cantidad de datos faltantes y del objetivo del análisis.

En este capítulo vamos a trabajar con tres estrategias generales:

```text
eliminar
imputar
reconstruir
```

**Eliminar** significa quitar filas o columnas que contienen valores faltantes. Es una estrategia simple, pero puede hacernos perder información. Si eliminamos demasiadas filas, el dataset puede quedar reducido o sesgado. Si eliminamos una columna completa, podemos perder una variable que tal vez era importante.

**Imputar** significa completar valores faltantes con algún valor elegido. Ese valor puede ser una constante, una categoría como `"Desconocido"`, una medida estadística como la media o la mediana, o algún valor definido según el contexto. Imputar permite conservar filas, pero también introduce información que no estaba originalmente en el dataset.

**Reconstruir** significa calcular un valor faltante a partir de otras columnas disponibles. Esta suele ser una estrategia más sólida que imputar de manera genérica, siempre que exista una relación confiable entre las columnas. En este dataset, por ejemplo, podemos usar la relación entre `Quantity`, `Price Per Unit` y `Total Spent`.

Podemos pensar estas estrategias de esta manera:

| Estrategia  | Qué hace                                   | Riesgo principal                             |
| ----------- | ------------------------------------------ | -------------------------------------------- |
| Eliminar    | Quita filas o columnas incompletas         | Perder información útil                      |
| Imputar     | Completa valores faltantes con un criterio | Introducir información artificial            |
| Reconstruir | Calcula valores usando otras columnas      | Depender de una relación que debe ser válida |

Antes de elegir una estrategia, necesitamos hacernos una pregunta fundamental:

```text
¿Qué representa este valor faltante y qué impacto tiene sobre el análisis?
```

La respuesta puede cambiar de una columna a otra. Por eso, en limpieza de datos, no conviene aplicar la misma solución a todo el dataset sin revisar primero el contexto.

## Eliminar filas con valores faltantes

Una primera estrategia posible es eliminar las filas que tienen valores faltantes.

En Pandas, podemos hacerlo con `dropna()`.

Si usamos `dropna()` sin parámetros adicionales, Pandas elimina todas las filas que tengan al menos un valor faltante.

In [ ]:
df_sin_faltantes = df.dropna()

df_sin_faltantes.shape

(4550, 8)

La salida muestra cuántas filas y columnas quedan después de eliminar las filas incompletas.

Para interpretar mejor el impacto de esta decisión, conviene comparar el tamaño del dataset original con el tamaño del dataset resultante.

In [ ]:
print("Tamaño original:", df.shape)
print("Tamaño después de dropna():", df_sin_faltantes.shape)

Tamaño original: (10000, 8)
Tamaño después de dropna(): (4550, 8)


Esta comparación es fundamental.

Eliminar filas con valores faltantes puede parecer una solución rápida, pero también puede reducir mucho el dataset. Si se eliminan muchas filas, podríamos perder información importante o modificar la composición de los datos.

Por eso, `dropna()` debe usarse con cuidado. No deberíamos aplicarlo automáticamente sin revisar cuántas filas se pierden y qué información contienen esas filas.

También podemos calcular cuántas filas fueron eliminadas:

In [ ]:
filas_eliminadas = df.shape[0] - df_sin_faltantes.shape[0]

filas_eliminadas

5450

Y podemos calcular qué porcentaje del dataset original representa esa pérdida:

In [ ]:
porcentaje_eliminado = filas_eliminadas / df.shape[0] * 100

porcentaje_eliminado

54.50000000000001

Si el porcentaje eliminado es alto, usar `dropna()` de manera general puede ser una mala decisión.

En algunos casos, eliminar filas incompletas puede ser razonable. Por ejemplo, si son muy pocas filas y los valores faltantes afectan columnas centrales que no pueden reconstruirse. Pero en otros casos puede ser preferible conservar las filas y aplicar otra estrategia.

La idea principal es que eliminar datos siempre tiene un costo: perdemos registros. Por eso, antes de hacerlo, debemos medir el impacto de esa decisión.

## Eliminar filas según columnas específicas

Usar `dropna()` sobre todo el `DataFrame` puede ser una decisión demasiado amplia. Tal vez una fila tenga un valor faltante en una columna secundaria, pero conserve información útil en columnas importantes.

Por eso, muchas veces conviene aplicar la eliminación solo considerando ciertas columnas.

Por ejemplo, podríamos decidir que para algunos análisis necesitamos conservar únicamente las filas que tienen información válida en estas columnas:

```text
Item
Quantity
Price Per Unit
Total Spent
```

Estas columnas son importantes porque permiten saber qué producto se vendió, cuántas unidades se vendieron, cuál fue el precio unitario y cuánto se gastó en total.

Podemos usar `dropna()` con el parámetro `subset` para indicar qué columnas deben revisarse.


In [ ]:
columnas_ventas = [
    "Item",
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

df_ventas_completas = df.dropna(subset=columnas_ventas)

df_ventas_completas.shape

(9197, 8)

En este caso, Pandas elimina solamente las filas que tienen valores faltantes en alguna de las columnas indicadas en `subset`.

Si una fila tiene un faltante en otra columna, como `Payment Method` o `Location`, no será eliminada por esta instrucción.

Comparemos nuevamente los tamaños:

In [ ]:
print("Tamaño original:", df.shape)
print("Tamaño con ventas completas:", df_ventas_completas.shape)

Tamaño original: (10000, 8)
Tamaño con ventas completas: (9197, 8)


Esta estrategia es más controlada que usar `dropna()` sobre todo el `DataFrame`.

La diferencia es importante: no estamos diciendo “eliminar cualquier fila con cualquier dato faltante”, sino “eliminar las filas que no tienen información suficiente en las columnas que consideramos centrales para este análisis”.

En limpieza de datos, este tipo de decisión suele ser más razonable porque se apoya en el significado de las columnas y no solo en la presencia general de valores faltantes.

## Eliminar columnas con demasiados valores faltantes

Además de eliminar filas, también podríamos eliminar columnas.

Esta decisión puede tener sentido cuando una columna tiene una proporción muy alta de valores faltantes y no resulta central para el análisis. Sin embargo, eliminar una columna completa es una decisión importante, porque implica descartar una variable del dataset.

Antes de eliminar columnas, conviene revisar el porcentaje de faltantes.

In [ ]:
diagnostico_faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": df.isna().mean() * 100
})

diagnostico_faltantes.sort_values("porcentaje", ascending=False)

,faltantes,porcentaje
Location,3265,32.65
Payment Method,2579,25.79
Item,333,3.33
Price Per Unit,179,1.79
Total Spent,173,1.73
Transaction Date,159,1.59
Quantity,138,1.38
Transaction ID,0,0.00


Supongamos que decidimos eliminar columnas con más del 50% de valores faltantes.

Para hacerlo, podemos usar `dropna()` con el parámetro `axis=1`.

Cuando `axis=0`, Pandas trabaja sobre filas. Cuando `axis=1`, trabaja sobre columnas.

También podemos usar el parámetro `thresh`, que indica cuántos valores no faltantes debe tener una columna para conservarse.

Por ejemplo, si queremos conservar solamente las columnas que tienen al menos la mitad de sus valores presentes, podemos calcular ese mínimo de esta manera:

In [ ]:
minimo_valores_presentes = len(df) * 0.5

minimo_valores_presentes

5000.0

In [ ]:
df_columnas_suficientes = df.dropna(
    axis=1,
    thresh=minimo_valores_presentes
)

df_columnas_suficientes.shape

(10000, 8)

Esta instrucción conserva solamente las columnas que tienen al menos la cantidad mínima de valores no faltantes indicada en `thresh`.

En este ejemplo usamos un umbral del 50%, pero ese número no debe tomarse como una regla universal. En algunos análisis podríamos tolerar más faltantes si la columna es muy importante. En otros casos, podríamos exigir columnas casi completas.

También podemos comparar qué columnas quedaron:

In [ ]:
print("Columnas originales:")
print(list(df.columns))

print()

print("Columnas conservadas:")
print(list(df_columnas_suficientes.columns))

Columnas originales:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Columnas conservadas:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


Eliminar columnas puede simplificar el dataset, pero también puede hacernos perder información valiosa. Por eso, esta decisión debe tomarse con cuidado.

Antes de eliminar una columna, conviene preguntarnos:

```text
¿Qué representa esta columna?
¿Cuántos valores faltan?
¿La columna es importante para el análisis?
¿Existe alguna forma razonable de completar o reconstruir sus valores?
¿Qué perdemos si la eliminamos?
```

La cantidad de faltantes es un dato importante, pero no debería ser el único criterio para decidir.

## Imputar valores faltantes con un valor fijo

Otra estrategia posible es completar valores faltantes con algún valor definido. A esta acción se la suele llamar **imputar**.

Imputar significa reemplazar un valor faltante por otro valor. Ese valor puede ser una constante, una categoría, una medida estadística o un valor calculado a partir de otras columnas.

Una forma simple de imputación es completar valores faltantes en una columna categórica con una etiqueta como `"Desconocido"`.

Por ejemplo, si faltan valores en la columna `Payment Method`, podríamos decidir conservar esas transacciones y registrar que el método de pago es desconocido.

Antes de aplicar el cambio, revisemos cuántos faltantes tiene esa columna.

In [ ]:
df["Payment Method"].isna().sum()

np.int64(2579)

Ahora podemos crear una copia del `DataFrame` y completar los valores faltantes de `Payment Method` con `"Desconocido"`.

Usamos una copia para no modificar todavía el dataset original.

In [ ]:
df_imputado = df.copy()

df_imputado["Payment Method"] = df_imputado["Payment Method"].fillna("Desconocido")

df_imputado["Payment Method"].value_counts(dropna=False)

,count
Payment Method,
Desconocido,2579
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


El método `fillna()` permite reemplazar valores faltantes por un valor determinado.

En este caso, los `NaN` de la columna `Payment Method` fueron reemplazados por `"Desconocido"`.

Esta estrategia puede ser razonable cuando el dato faltante representa una categoría desconocida y queremos conservar las filas para otros análisis.

Sin embargo, también tiene un riesgo: al completar los faltantes con una etiqueta, estamos creando una categoría nueva que no estaba explícitamente registrada en los datos originales. Eso puede ser útil, pero debe interpretarse correctamente.

No significa que el método de pago haya sido realmente `"Desconocido"` como una opción elegida por el cliente. Significa que nosotros decidimos etiquetar así las transacciones donde esa información no estaba disponible.

Antes de imputar valores numéricos, necesitamos revisar un detalle importante de este dataset. Algunas columnas que deberían ser numéricas fueron cargadas como texto, porque contienen valores como `"ERROR"` o `"UNKNOWN"` mezclados con números.

Más adelante vamos a estudiar la conversión de tipos de datos con mayor profundidad. Por ahora, para poder trabajar con imputación y reconstrucción de valores, vamos a crear una copia del dataset y convertir algunas columnas a formato numérico.

Usaremos `pd.to_numeric()` con `errors="coerce"`. Esto intenta convertir los valores a número y, cuando encuentra un valor que no puede convertir, lo transforma en `NaN`.

In [ ]:
df_tratamiento = df.copy()

columnas_numericas = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for columna in columnas_numericas:
    df_tratamiento[columna] = pd.to_numeric(
        df_tratamiento[columna],
        errors="coerce"
    )

df_tratamiento[columnas_numericas].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Quantity        9521 non-null   float64
 1   Price Per Unit  9467 non-null   float64
 2   Total Spent     9498 non-null   float64
dtypes: float64(3)
memory usage: 234.5 KB


## Imputar valores numéricos con una medida estadística

Cuando una columna numérica tiene valores faltantes, una estrategia posible es completarlos con una medida estadística.

Por ejemplo, podríamos usar la media, la mediana o algún otro valor representativo de la columna.

Esta estrategia puede parecer sencilla, pero debe usarse con cuidado. Completar valores faltantes con una medida estadística introduce información artificial en el dataset. No estamos recuperando el dato real, sino reemplazándolo por una estimación.

Por eso, antes de imputar una columna numérica conviene revisar su distribución y pensar si la medida elegida tiene sentido.

Veamos un ejemplo con la columna `Price Per Unit`.

In [ ]:
df_tratamiento["Price Per Unit"].isna().sum()

np.int64(533)

Primero revisamos cuántos valores faltantes tiene la columna.

Ahora podemos observar algunas estadísticas básicas:

In [ ]:
df_tratamiento["Price Per Unit"].describe()

,Price Per Unit
count,9467.000000
mean,2.949984
std,1.278450
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000


Si decidiéramos completar los valores faltantes con la mediana, podríamos hacerlo así:

In [ ]:
df_precio_imputado = df_tratamiento.copy()

mediana_precio = df_precio_imputado["Price Per Unit"].median()

df_precio_imputado["Price Per Unit"] = (
    df_precio_imputado["Price Per Unit"].fillna(mediana_precio)
)

df_precio_imputado["Price Per Unit"].isna().sum()

np.int64(0)

Después de aplicar `fillna()`, la columna ya no tiene valores faltantes reconocidos por Pandas.

Sin embargo, esto no significa que hayamos recuperado los precios reales que faltaban. Lo que hicimos fue reemplazar esos valores por la mediana de la columna.

La mediana puede ser una opción razonable cuando queremos evitar que valores muy altos o muy bajos influyan demasiado en la imputación. A diferencia de la media, la mediana suele ser más resistente a valores extremos.

Aun así, esta decisión debe justificarse. No deberíamos completar valores numéricos automáticamente solo porque Pandas nos permite hacerlo.

En algunos casos, imputar con una medida estadística puede ser útil. En otros, puede ocultar problemas importantes o distorsionar el análisis.

## Reconstruir valores usando otras columnas

En algunos casos, no necesitamos imputar un valor faltante con una media, una mediana o una etiqueta fija. A veces podemos reconstruirlo usando información disponible en otras columnas.

Este dataset tiene una relación muy importante:

```text
Total Spent = Quantity × Price Per Unit
```

Esto significa que, si conocemos la cantidad y el precio unitario, podemos calcular el total gastado.

Esta estrategia suele ser más razonable que imputar con una medida general, porque utiliza una relación interna del propio dataset.

Primero observemos cuántos valores faltantes tiene cada una de estas columnas.

In [ ]:
df_tratamiento[["Quantity", "Price Per Unit", "Total Spent"]].isna().sum()

,0
Quantity,479
Price Per Unit,533
Total Spent,502


Ahora vamos a buscar filas donde falta `Total Spent`, pero sí están presentes `Quantity` y `Price Per Unit`.

En esos casos, podríamos reconstruir el total.

In [ ]:
condicion_reconstruir_total = (
    df_tratamiento["Total Spent"].isna()
    & df_tratamiento["Quantity"].notna()
    & df_tratamiento["Price Per Unit"].notna()
)

df_tratamiento[condicion_reconstruir_total][
    ["Quantity", "Price Per Unit", "Total Spent"]
].head()

,Quantity,Price Per Unit,Total Spent
2,4.0,1.0,NaN
25,3.0,4.0,NaN
31,2.0,1.0,NaN
42,2.0,1.5,NaN
94,3.0,3.0,NaN


La condición anterior identifica filas donde `Total Spent` está faltante, pero las otras dos columnas necesarias para calcularlo están disponibles.

Para reconstruir esos valores, vamos a trabajar sobre una copia del dataset.

In [ ]:
df_reconstruido = df_tratamiento.copy()

df_reconstruido.loc[
    condicion_reconstruir_total,
    "Total Spent"
] = (
    df_reconstruido.loc[condicion_reconstruir_total, "Quantity"]
    * df_reconstruido.loc[condicion_reconstruir_total, "Price Per Unit"]
)

df_reconstruido[condicion_reconstruir_total][
    ["Quantity", "Price Per Unit", "Total Spent"]
].head()

,Quantity,Price Per Unit,Total Spent
2,4.0,1.0,4.0
25,3.0,4.0,12.0
31,2.0,1.0,2.0
42,2.0,1.5,3.0
94,3.0,3.0,9.0


Ahora las filas seleccionadas tienen un valor reconstruido en `Total Spent`.

Esta estrategia no consiste en inventar un valor promedio, sino en aprovechar una relación esperada entre columnas. Por eso puede ser más defendible que una imputación estadística genérica.

De todos modos, también requiere cuidado. Solo podemos reconstruir valores si estamos seguros de que la relación entre columnas es válida y si las columnas usadas para el cálculo son confiables.

Por eso, después de reconstruir valores, siempre debemos verificar el resultado.

## Verificar valores reconstruidos

Después de reconstruir valores faltantes, necesitamos verificar que la transformación haya producido resultados coherentes.

En este caso reconstruimos algunos valores de `Total Spent` usando la relación:

```text
Total Spent = Quantity × Price Per Unit
```

Podemos comprobar si, para las filas reconstruidas, el valor de `Total Spent` coincide con la multiplicación entre `Quantity` y `Price Per Unit`.

In [ ]:
df_reconstruido.loc[
    condicion_reconstruir_total,
    ["Quantity", "Price Per Unit", "Total Spent"]
].head()

,Quantity,Price Per Unit,Total Spent
2,4.0,1.0,4.0
25,3.0,4.0,12.0
31,2.0,1.0,2.0
42,2.0,1.5,3.0
94,3.0,3.0,9.0


También podemos crear una comparación explícita para esas filas.

In [ ]:
verificacion_total = (
    df_reconstruido.loc[condicion_reconstruir_total, "Quantity"]
    * df_reconstruido.loc[condicion_reconstruir_total, "Price Per Unit"]
)

verificacion_total.head()

,0
2,4.0
25,12.0
31,2.0
42,3.0
94,9.0


Ahora podemos comparar ese cálculo con la columna `Total Spent` reconstruida.

In [ ]:
(
    df_reconstruido.loc[condicion_reconstruir_total, "Total Spent"]
    == verificacion_total
).value_counts()

,count
True,462


Si la salida muestra `True` para todas las filas reconstruidas, significa que los valores cargados en `Total Spent` coinciden con el cálculo esperado.

Esta verificación es importante porque la reconstrucción no termina en la asignación. También necesitamos comprobar que el resultado respete la relación que usamos para completarlo.

En un proceso real de limpieza, cada transformación importante debería tener una verificación posterior. Esa verificación puede ser un conteo, una comparación, una revisión de filas, un resumen estadístico o cualquier otra comprobación que nos permita confiar en el cambio realizado.

## Comparar estrategias de tratamiento

Hasta ahora vimos tres formas posibles de tratar valores faltantes:

```text
eliminar
imputar
reconstruir
```

Cada una responde a una lógica diferente.

Eliminar datos puede ser útil cuando los registros incompletos son pocos o cuando no hay una forma razonable de recuperar la información. Pero también puede reducir el tamaño del dataset y modificar su composición.

Imputar permite conservar registros, pero introduce un valor que no estaba originalmente en los datos. Puede ser una etiqueta como `"Desconocido"` o una medida estadística como la mediana. Esta estrategia debe interpretarse con cuidado, porque no recupera el dato real.

Reconstruir valores suele ser una alternativa más sólida cuando existe una relación confiable entre columnas. En este dataset, por ejemplo, `Total Spent` puede reconstruirse si conocemos `Quantity` y `Price Per Unit`.

Podemos resumirlo así:

| Estrategia                     | Cuándo puede servir                                                                  | Qué debemos verificar                                             |
| ------------------------------ | ------------------------------------------------------------------------------------ | ----------------------------------------------------------------- |
| Eliminar filas                 | Cuando los faltantes son pocos o afectan columnas críticas imposibles de reconstruir | Cuántas filas se pierden y si la pérdida puede sesgar el análisis |
| Eliminar columnas              | Cuando una columna tiene demasiados faltantes y no es central                        | Qué información se pierde al quitar esa variable                  |
| Imputar con valor fijo         | Cuando queremos conservar registros y marcar ausencia de información                 | Que el nuevo valor no se interprete como dato real observado      |
| Imputar con medida estadística | Cuando una estimación resulta razonable para una columna numérica                    | Que la imputación no distorsione la distribución                  |
| Reconstruir                    | Cuando otras columnas permiten calcular el valor faltante                            | Que la relación usada para reconstruir sea válida                 |

La decisión depende del contexto. No deberíamos aplicar una misma estrategia a todas las columnas solo porque es rápida o sencilla.

En limpieza de datos, una buena decisión no es necesariamente la que elimina más problemas visibles, sino la que conserva mejor la información útil y deja el dataset más confiable para el análisis.

## Trabajar con copias durante la limpieza

Cuando estamos probando estrategias de tratamiento, conviene evitar modificar el dataset original de manera apresurada.

Por eso, en varios ejemplos usamos copias:

```python
df_imputado = df.copy()
```

o:

```python
df_reconstruido = df.copy()
```

Esta práctica nos permite experimentar con una transformación sin perder el punto de partida. Si la decisión no resulta adecuada, podemos volver al dataset original y probar otra estrategia.

Trabajar con copias también ayuda a comparar resultados. Por ejemplo, podemos revisar cuántos valores faltantes tenía el dataset original y cuántos quedan después de una imputación o reconstrucción.

Veamos un ejemplo comparando `df` y `df_reconstruido` en la columna `Total Spent`.


In [ ]:
print("Faltantes originales en Total Spent:")
print(df["Total Spent"].isna().sum())

print()

print("Faltantes después de reconstruir Total Spent:")
print(df_reconstruido["Total Spent"].isna().sum())

Faltantes originales en Total Spent:
173

Faltantes después de reconstruir Total Spent:
40


Esta comparación muestra si la reconstrucción redujo la cantidad de valores faltantes en esa columna.

También podemos comparar el tamaño del dataset original con el de una versión donde eliminamos filas incompletas:

In [ ]:
print("Tamaño original:")
print(df.shape)

print()

print("Tamaño después de eliminar filas con faltantes:")
print(df_sin_faltantes.shape)

Tamaño original:
(10000, 8)

Tamaño después de eliminar filas con faltantes:
(4550, 8)


Estas comparaciones permiten documentar el impacto de cada decisión.

En limpieza de datos, no alcanza con transformar el dataset. También necesitamos poder explicar qué hicimos, por qué lo hicimos y qué cambió después de hacerlo.

Una buena práctica es conservar evidencia del proceso:

```text
conteos antes y después
porcentajes antes y después
cantidad de filas eliminadas
cantidad de valores imputados
cantidad de valores reconstruidos
```

Esto vuelve el proceso más transparente y facilita revisar las decisiones tomadas.


## Errores frecuentes al tratar valores faltantes

Al tratar valores faltantes, uno de los errores más comunes es aplicar una solución automática sin diagnóstico previo.

Por ejemplo, podríamos usar `dropna()` sobre todo el `DataFrame` y eliminar todas las filas incompletas. Técnicamente es una instrucción válida, pero puede ser una mala decisión si elimina una parte importante del dataset o si descarta registros que todavía eran útiles para otros análisis.

Otro error frecuente es imputar valores sin justificar el criterio. Completar una columna numérica con la media o la mediana puede ser útil en algunos casos, pero no debe hacerse solo porque es rápido. Esa decisión puede modificar la distribución de los datos y afectar los resultados posteriores.

También debemos evitar tratar todos los faltantes como si fueran iguales. Un valor faltante en `Payment Method` no tiene el mismo impacto que un faltante en `Quantity`, `Price Per Unit` o `Total Spent`. Cada columna debe analizarse según lo que representa.

Un caso especialmente importante es confundir imputación con reconstrucción. Si completamos un valor usando una mediana, estamos estimando. Si lo calculamos a partir de una relación confiable entre columnas, estamos reconstruyendo. Ambas acciones completan un faltante, pero no tienen el mismo significado.

También puede ocurrir que tratemos solamente los `NaN` y olvidemos valores como `"UNKNOWN"` o `"ERROR"`. Esos valores pueden representar ausencia de información o errores de carga, aunque Pandas los interprete como texto común. Por eso, el tratamiento de faltantes no debe limitarse siempre a `isna()`.

Finalmente, otro error frecuente es no verificar después de transformar. Después de eliminar, imputar o reconstruir datos, debemos revisar qué cambió.

In [ ]:
# Ejemplo de verificación posterior para valores faltantes reconocidos por Pandas.

df_reconstruido.isna().sum()

,0
Transaction ID,0
Item,333
Quantity,479
Price Per Unit,533
Total Spent,40
Payment Method,2579
Location,3265
Transaction Date,159


Esta revisión permite ver cuántos valores faltantes quedan después de la reconstrucción realizada.

En general, al tratar valores faltantes conviene seguir esta rutina:

```text
diagnosticar
decidir una estrategia
aplicar la transformación
verificar el resultado
documentar la decisión
```

El objetivo no es dejar el dataset “sin ningún faltante” a cualquier costo. El objetivo es tomar decisiones razonables para que el dataset quede más preparado y más confiable para el análisis.

## Resumen del capítulo

En este capítulo trabajamos con distintas estrategias para tratar valores faltantes.

Partimos de una idea central: detectar valores faltantes no alcanza. Después del diagnóstico necesitamos decidir qué hacer con ellos. Esa decisión depende del contexto, de la columna afectada, de la cantidad de datos incompletos y del tipo de análisis que queremos realizar.

Primero distinguimos tres estrategias generales:

```text
eliminar
imputar
reconstruir
```

Eliminar significa quitar filas o columnas con valores faltantes. Usamos `dropna()` para eliminar filas incompletas:

```python
df_sin_faltantes = df.dropna()
```

Después comparamos el tamaño del dataset original con el tamaño del dataset resultante para medir cuántas filas se perdían. Esta comparación fue importante porque eliminar datos siempre tiene un costo: reduce el tamaño del dataset y puede modificar su composición.

También vimos una forma más controlada de eliminar filas usando `subset`:

```python id="ucbx0h"
df_ventas_completas = df.dropna(subset=[
    "Item",
    "Quantity",
    "Price Per Unit",
    "Total Spent"
])
```

En ese caso, no eliminamos cualquier fila con cualquier faltante, sino solamente aquellas que tenían datos incompletos en columnas consideradas centrales para cierto análisis.

Luego analizamos la posibilidad de eliminar columnas con muchos faltantes usando `dropna(axis=1, thresh=...)`. Vimos que esta decisión puede simplificar el dataset, pero también puede hacernos perder variables importantes.

Más adelante trabajamos con imputación. Usamos `fillna()` para completar valores faltantes con un valor fijo:

```python id="6fkbrh"
df_imputado["Payment Method"] = (
    df_imputado["Payment Method"].fillna("Desconocido")
)
```

También vimos un ejemplo de imputación numérica usando la mediana:

```python id="siz5go"
mediana_precio = df_imputado["Price Per Unit"].median()

df_imputado["Price Per Unit"] = (
    df_imputado["Price Per Unit"].fillna(mediana_precio)
)
```

En ambos casos aclaramos que imputar no significa recuperar el dato real. Significa completar un faltante con un criterio elegido. Por eso, la imputación debe interpretarse con cuidado.

Después trabajamos con una estrategia más sólida cuando el dataset lo permite: reconstruir valores usando otras columnas. En este caso usamos la relación:

```text
Total Spent = Quantity × Price Per Unit
```

A partir de esa relación, reconstruimos valores faltantes de `Total Spent` cuando `Quantity` y `Price Per Unit` estaban disponibles.

También verificamos la reconstrucción comparando el valor reconstruido con el cálculo esperado. Esa verificación fue clave para confirmar que la transformación se aplicó correctamente.

Finalmente, comparamos las estrategias y reforzamos la importancia de trabajar con copias durante la limpieza. Usar `df.copy()` nos permite probar transformaciones, comparar resultados y conservar el dataset original como punto de referencia.

La idea principal de este capítulo fue:

```text
Tratar valores faltantes implica decidir entre eliminar, imputar o reconstruir, según el contexto y el impacto de cada decisión.
```

No existe una única estrategia correcta para todos los casos. Una limpieza responsable debe diagnosticar, decidir, transformar, verificar y documentar.

## Próximo paso

El siguiente paso será trabajar con duplicados.

En los datasets reales puede ocurrir que una misma observación aparezca más de una vez. A veces se trata de un error de carga o una repetición accidental. Otras veces, en cambio, dos filas parecidas pueden representar eventos reales distintos.

Vamos a aprender a detectar duplicados, contar cuántos hay, observarlos, decidir si corresponde eliminarlos y verificar el resultado después de hacerlo.